# a_Get_Exp_Data Runner

`run_get_exp_data.py`를 노트북에서 셀 단위로 실행하기 위한 작업 노트북입니다.

- 단일 피험자 / 다중 피험자 실행
- `dry_run` 점검
- 조건별 디버깅 전 실행

In [1]:
import os
import sys
import matplotlib

# 노트북 실행 위치가 어디든 동작하도록 경로를 정렬
repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "Codes")): # if you want to run this notebook from other directory, change the path below
    repo_root = r"C:/Users/ok/Documents/GitHub/BOX"     # (optional)TODO: change to your path

work_dir = os.path.join(repo_root, "Codes", "a_Get_Exp_Data")
os.chdir(work_dir)

if work_dir not in sys.path:
    sys.path.insert(0, work_dir)
codes_dir = os.path.dirname(work_dir)
if codes_dir not in sys.path:
    sys.path.insert(0, codes_dir)

print("cwd:", os.getcwd())
print("work_dir:", work_dir)


import PATH_RULE as _path
from run_get_exp_data import process_subject

print("Available subjects:", _path.DATA_SUB_NAMECODE_li)

cwd: C:\Users\ok\Documents\GitHub\BOX\Codes\a_Get_Exp_Data
work_dir: C:/Users/ok/Documents/GitHub/BOX\Codes\a_Get_Exp_Data
Available subjects: ['240124_PJH', '260306_KTY', '260423_CES', '260512_KCH']


In [2]:
# 단일 피험자 실제 실행
dry_run = True
namecode = "260512_KCH"  # TODO: 필요 시 변경
process_subject(namecode, dry_run=dry_run, t_tap_offset=-2.3)   # t_tap_offset: 피험자별 탭핑 시점 조정 값. 음수면 앞당기기.


[260512_KCH]  SUB4  protocol=Asymmetric  method=bpm_window
  C3D dir : E:\Dropbox\SEL\BOX\Experiment\260512_KCH\Labeled
  Rigid dir: E:\Dropbox\SEL\BOX\Experiment\260512_KCH\RigidBody
  Output  : C:\Users\ok\Documents\GitHub\BOX\OpenSim_Process\_Main_\Asymmetric\SUB4
  APPs    : ['MeasuredEHF', 'HeavyHand', 'preRiCTO', 'postRiCTO']
  t_tap_offset : -2.30s  (manual)
  [static]  OK  Static.c3d  →  C:\Users\ok\Documents\GitHub\BOX\OpenSim_Process\_Main_\Asymmetric\SUB4\Model_osim\static.trc

  [7kg_10bpm]  C3D=OK  Rigid=OK  cycles=10
    C3D  : 7kg_10bpm.c3d
    Rigid: 7kg_10bpm_rigidbody.csv
    ── dry-run plan ──
    C3D candidates (1):
      7kg_10bpm.c3d  <- selected
    Rigid candidates (1):
      7kg_10bpm_rigidbody.csv  <- selected
    BPM extracted: 10
    Window duration: 6.0s
    Planned outputs: 30 TRC + 60 MOT (2/4 APPs implemented)
      section AB: ['1AB', '2AB', '3AB', '4AB', '5AB', '6AB', '7AB', '8AB', '9AB', '10AB']
      section BC: ['1BC', '2BC', '3BC', '4BC', '5BC', '

In [ ]:
# 단일 피험자 dry-run 점검
namecode = "240124_PJH"  # TODO: 필요 시 변경
process_subject(namecode, dry_run=True)

In [ ]:
# 여러 피험자 일괄 실행 (옵션)
dry_run = True
subject_list = [
    # "240124_PJH",
    "260306_KTH",
    
]

for nc in subject_list:
    process_subject(nc, dry_run=dry_run)

In [4]:
# [GUI 매뉴얼 tap 선택]  bpm_window 의 t_tap 을 matplotlib 창에서 직접 클릭
#
# 주의: Jupyter 의 기본 backend (%matplotlib inline) 는 GUI 창을 띄울 수 없으므로
#       아래 매직으로 별도 창을 띄우는 backend 로 전환해야 한다.
#       - 환경에 따라 둘 중 하나 사용: `%matplotlib qt`  또는  `%matplotlib tk`
#       - 한 번 전환하면 커널 재시작 전까지 유지됨.
#
# 단축키:
#   좌클릭         : t_tap 위치 설정 (0.01s 그리드로 스냅)
#   ← / →          : ±0.01s 미세 조정
#   Shift + ← / →  : ±0.1s 조정
#   r              : 자동 검출값으로 reset
#   Enter / 창 닫기: 현재 선택값 확정 → 다음 condition 으로 진행
#
# t_tap_offset 은 GUI 의 "초기 선택 위치" 로 사용된다 (없으면 자동 검출값).
# scalar 또는 dict 두 가지 형태 지원:
#   - scalar  : 모든 condition 에 동일 적용              → t_tap_offset=-2.3
#   - dict    : condition 별로 다른 값                   → t_tap_offset={"7kg_10bpm": -2.3, "7kg_16bpm": -1.5}
#     · dict 에 없는 cond_key 는 "_default" 값으로 폴백  → t_tap_offset={"_default": -2.0, "7kg_16bpm": -1.5}
#     · "_default" 도 없으면 0.0 으로 폴백 (정보 print)

# %matplotlib qt
%matplotlib tk

dry_run  = True            # True: tap_onset_check.png 만 저장,  False: 실제 TRC/MOT 생성
namecode = "260512_KCH"    # TODO: 필요 시 변경

# condition 별로 다른 초기 offset 을 주는 예시.
# (인터랙티브 1회 후 콘솔에 출력되는 effective offset 을 채워넣고 재실행하면
#  동일한 결과를 자동으로 재현할 수 있음.)
t_tap_offset_per_cond = {
    # "_default":  -2.00,   # 명시 안 한 cond 에 적용할 기본값 (선택)
    "7kg_10bpm":  -2.30,
    # "7kg_16bpm": -1.50,
    # "15kg_10bpm": -2.45,
}

process_subject(
    namecode,
    dry_run=dry_run,
    interactive_tap=True,
    t_tap_offset=t_tap_offset_per_cond,
)


ImportError: Failed to import any of the following Qt binding modules: PyQt6, PySide6, PyQt5, PySide2